In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

bronze_path = "s3://incremental-load-csv/bronze/"
gold_path = "s3://incremental-load-csv/gold/"
parquet_gold_path = "s3://incremental-load-csv/gold-parquet/"
tracker_path = "s3://incremental-load-csv/checkpoints/file_tracker/"


# Get new file

all_files = [f.path for f in dbutils.fs.ls(bronze_path)]

#Only take CSV files
all_files = [f for f in all_files if f.endswith(".csv")]

#Load processed files
try:
    processed_df = spark.read.parquet(tracker_path)
    processed_files = [row.file_name for row in processed_df.collect()]
except:
    processed_files = []

#find new files
new_files = list(set(all_files) - set(processed_files))

print("new files:", new_files)

#Stop if no New File Detected
if len(new_files) == 0:
    print("no new files to process")
    dbutils.notebook.exit("No new data")

#Read bronze data
schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("genre", StringType(), True),
    StructField("watch_time_minutes", IntegerType(), True),
    StructField("platform", StringType(), True),
    StructField("country", StringType(), True),
    StructField("event_date", StringType(), True)
])

bronze_df = spark.read.csv(new_files, header=True, schema=schema)

bronze_df = bronze_df.withColumn("ingestion_time", current_timestamp())


#Silver transformation
silver_df = (
    bronze_df.dropDuplicates(["event_id"]).filter(col("watch_time_minutes") > 0).withColumn("event_date",to_date("event_date"))
)

#Gold aggregation
gold_df = (
    silver_df.groupBy("event_date","platform").agg(
        sum("watch_time_minutes").alias("total_watch_time"),countDistinct("user_id").alias("active_users")
    )
)

#Merge into gold (single table)
try:
    gold_table = DeltaTable.forPath(spark, gold_path)

    (gold_table.alias("t").merge(gold_df.alias("s"),"t.event_date = s.event_date AND t.platform = s.platform").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
except:
    gold_df.write.format("delta").mode("overwrite").save(gold_path)


# Gold data in parquet for athena query
gold_full_df = spark.read.format("delta").load(gold_path)

gold_full_df.write \
    .mode("overwrite") \
    .parquet(parquet_gold_path)


# Update processed file tracker

new_files_df = spark.createDataFrame([(f,) for f in new_files], ["file_name"])

try:
    existing = spark.read.parquet(tracker_path)
    updated = existing.union(new_files_df)
except:
    updated = new_files_df

#updated.write.mode("overwrite").parquet(tracker_path)


